# Chemistry domain adaptation with Phi-3 and LoRA

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/corndel-ai/LLM-fine-tune/blob/main/chemistry_llm_colab.ipynb)

This notebook performs a small **domain-adaptive pretraining (DAPT)** exercise on OECD chemistry test guidelines. In DAPT, a pretrained language model continues learning from unlabelled domain text by predicting the next token. This is different from supervised fine-tuning (SFT), which trains on prompt/answer examples to teach a behaviour. **LoRA** is the parameter-efficient mechanism used here: most model weights stay frozen while small trainable updates are added to its linear layers.

Lower loss means the model became better at predicting held-out text from this small corpus; it does **not** prove that answers are correct, safe, or OECD-endorsed. Model card: https://huggingface.co/microsoft/Phi-3-mini-4k-instruct

The included PDFs are third-party OECD materials, not MIT-licensed repository code. Review their source and reuse notes in [OECD_MATERIALS.md](https://github.com/corndel-ai/LLM-fine-tune/blob/main/OECD_MATERIALS.md) and the current OECD terms: https://www.oecd.org/en/about/terms-conditions.html

> In Colab, first select an NVIDIA GPU runtime, then choose **Runtime > Run all**. No Drive mount, token, upload, or prompt is required.

## 1. Install the notebook dependencies

We retain Colab's compatible PyTorch/CUDA build and install explicit versions of the training libraries.

In [ ]:
%pip install -q --upgrade accelerate==1.15.0 bitsandbytes==0.50.2 datasets==5.0.1 fsspec==2025.12.0 huggingface-hub==1.31.0 jedi==0.20.0 peft==0.20.0 PyMuPDF==1.28.2 transformers==5.16.1
%pip check

## 2. Preflight the actual runtime

The experiment requires an NVIDIA CUDA GPU. Colab runtime guidance: https://research.google.com/colaboratory/runtime-version-faq.html and bitsandbytes support: https://huggingface.co/docs/bitsandbytes/installation

In [ ]:
import importlib.metadata as metadata
import json
import platform
import random
import shutil
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import torch

RUN_STARTED_AT = datetime.now(timezone.utc).isoformat()
PACKAGE_NAMES = ["accelerate", "bitsandbytes", "datasets", "fsspec", "huggingface-hub", "jedi", "peft", "PyMuPDF", "transformers"]
PACKAGE_VERSIONS = {name: metadata.version(name) for name in PACKAGE_NAMES}

print(f"Python: {sys.version.split()[0]} ({platform.platform()})")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()} | CUDA build: {torch.version.cuda}")
print(f"Packages: {PACKAGE_VERSIONS}")
print(f"Free disk at /content: {shutil.disk_usage('/content').free / 2**30:.1f} GiB")
assert torch.cuda.is_available(), "Select a GPU runtime in Colab, then run all cells again."

gpu = torch.cuda.get_device_properties(0)
free_bytes, total_bytes = torch.cuda.mem_get_info()
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"GPU: {gpu.name}")
print(f"Compute capability: {gpu.major}.{gpu.minor}")
print(f"GPU memory: {free_bytes / 2**30:.1f} GiB free / {total_bytes / 2**30:.1f} GiB total")
print(f"Training compute dtype: {COMPUTE_DTYPE}")
if total_bytes < 14 * 2**30:
    print("Warning: less than 14 GiB VRAM may cause an out-of-memory error.")

## 3. One visible experiment configuration

Keeping the important choices together makes the experiment easier to read, repeat, and alter. `RESUME_FROM_CHECKPOINT` may be `None`, `"latest"`, or a checkpoint path. The Trainer saves up to two checkpoints and reloads the one with the lowest validation loss, which helps recover an expensive run and avoids losing the strongest observed checkpoint.

In [ ]:
SEED = 42
MODEL_ID = "microsoft/Phi-3-mini-4k-instruct"
MODEL_REVISION = "f39ac1d28e925b323eae81227eaba4464caced4e"
BLOCK_SIZE = 512
MAX_STEPS = 20
EVAL_STEPS = 10  # The 20-step quick run has two checks; use 5 for patience-based early stopping.
LEARNING_RATE = 2e-4
TRAIN_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 2
LORA_RANK = 16
LORA_ALPHA = 32
EARLY_STOPPING_PATIENCE = 2
EARLY_STOPPING_THRESHOLD = 0.01
DOWNLOAD_ADAPTER = False
RESUME_FROM_CHECKPOINT = None
REPO_URL = "https://github.com/corndel-ai/LLM-fine-tune"
REPO_REF = "main"
print({name: value for name, value in globals().copy().items() if name in {
    "SEED", "MODEL_ID", "MODEL_REVISION", "BLOCK_SIZE", "MAX_STEPS",
    "EVAL_STEPS", "LEARNING_RATE", "TRAIN_BATCH_SIZE",
    "GRADIENT_ACCUMULATION_STEPS", "LORA_RANK", "LORA_ALPHA",
    "EARLY_STOPPING_PATIENCE", "EARLY_STOPPING_THRESHOLD",
    "DOWNLOAD_ADAPTER", "RESUME_FROM_CHECKPOINT", "REPO_URL", "REPO_REF"}})

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Reproducibility has hardware-dependent limits: https://docs.pytorch.org/docs/stable/notes/randomness.html

## 4. Download the PDFs directly from their raw URLs



In [ ]:
from urllib.request import urlopen, urlretrieve

PDF_FILENAMES = [
    "test-guideline-102.pdf", "test-guideline-104.pdf",
    "test-guideline-105.pdf", "test-guideline-109.pdf",
    "test-guideline-110.pdf", "test-guideline-111.pdf",
    "test-guideline-114.pdf", "test-guideline-117.pdf",
    "test-guideline-124.pdf", "test-guideline-125.pdf",
    "test-guideline-126.pdf",
]
COMMIT_API_URL = f"https://api.github.com/repos/corndel-ai/LLM-fine-tune/commits/{REPO_REF}"
with urlopen(COMMIT_API_URL) as response:
    REPO_COMMIT = json.load(response)["sha"]
RAW_PDF_BASE = (
    f"https://raw.githubusercontent.com/corndel-ai/LLM-fine-tune/{REPO_COMMIT}"
    "/oecd-pdfs/oecd-pdfs"
)
PDF_DIR = Path("/content/oecd-pdfs")
PDF_DIR.mkdir(parents=True, exist_ok=True)

for filename in PDF_FILENAMES:
    raw_url = f"{RAW_PDF_BASE}/{filename}"
    urlretrieve(raw_url, PDF_DIR / filename)
    print(f"Downloaded {raw_url}")

print(f"\nResolved repository commit: {REPO_COMMIT}")
print(f"Downloaded {len(PDF_FILENAMES)} PDFs to {PDF_DIR}")

## 5. Extract the PDFs page by page

Page-level extraction preserves useful provenance. PyMuPDF's sorted text option usually gives a more natural reading order than the PDF's internal object order: https://pymupdf.readthedocs.io/en/latest/recipes-text.html

In [ ]:
import hashlib

import pymupdf
import pandas as pd
from IPython.display import display

pdf_paths = [PDF_DIR / filename for filename in PDF_FILENAMES]
assert all(path.is_file() for path in pdf_paths), "One or more PDF downloads are missing."
page_records = []
inventory = []
pdf_hashes = {}

for pdf_path in pdf_paths:
    pdf_hashes[pdf_path.name] = hashlib.sha256(pdf_path.read_bytes()).hexdigest()
    with pymupdf.open(pdf_path) as document:
        first_page_text = document[0].get_text("text", sort=True) if document.page_count else ""
        title_lines = [line.strip() for line in first_page_text.splitlines() if line.strip()]
        inferred_title = " | ".join(title_lines[:4])[:180]
        inventory.append({
            "filename": pdf_path.name,
            "size_MiB": round(pdf_path.stat().st_size / 2**20, 2),
            "pages": document.page_count,
            "title_preview": inferred_title,
        })
        for page_index, page in enumerate(document):
            page_records.append({
                "source": pdf_path.name,
                "page_number": page_index + 1,
                "raw_text": page.get_text("text", sort=True),
            })

inventory_df = pd.DataFrame(inventory)
display(inventory_df)
print(f"Extracted {len(page_records):,} pages from {len(pdf_paths)} PDFs.")

### Look at raw extracted text

The rendered preview is easy to read; `repr` exposes hidden newlines and spacing that matter during cleaning.

In [ ]:
raw_example = next(record for record in page_records if record["raw_text"].strip())
print(f"Example: {raw_example['source']}, page {raw_example['page_number']}")
print("\nRendered preview:\n")
print(raw_example["raw_text"][:1200])
print("\nPython repr preview:\n")
print(repr(raw_example["raw_text"][:500]))

## 6. Clean conservatively and make the changes visible

PDFs carry layout artefacts that are not language: repeating OECD headers, copyright and reuse notices, and unambiguous page markers. The cleaner removes those and deliberately does very little else. It preserves standalone numbers because PDF table extraction can place scientific values on their own lines, and it keeps line-ending hyphens when joining wrapped text. It does not repair tables, reorder text, or guarantee that linearised PDF text retains the original visual structure.

In [ ]:
import re

NOISE_PATTERNS = [
    re.compile(r"^\s*\d+\s*/\s*\d+\s*$"),
    re.compile(r"^\s*OECD/OCDE(?:\s+\d+)?\s*$", re.IGNORECASE),
    re.compile(r"^\s*©\s*OECD.*$", re.IGNORECASE),
    re.compile(r"^\s*You are free to use this material.*$", re.IGNORECASE),
    re.compile(r"^\s*(?:Any commercial use|Permission from the OECD).*$", re.IGNORECASE),
]

def clean_page(raw_text):
    joined = re.sub(r"(?<=\w)-\s*\n\s*(?=\w)", "-", raw_text)
    kept_lines, removed_lines = [], []
    for line in joined.splitlines():
        line = re.sub(r"[ \t]+", " ", line).strip()
        if line and any(pattern.match(line) for pattern in NOISE_PATTERNS):
            removed_lines.append(line)
        elif line:
            kept_lines.append(line)
    cleaned = re.sub(r"\s+", " ", " ".join(kept_lines)).strip()
    return cleaned, removed_lines

cleaned_pages = []
all_removed_lines = []
for record in page_records:
    cleaned_text, removed_lines = clean_page(record["raw_text"])
    if cleaned_text:
        cleaned_pages.append({
            "source": record["source"],
            "page_number": record["page_number"],
            "text": cleaned_text,
        })
    all_removed_lines.extend(removed_lines)

raw_chars = sum(len(record["raw_text"]) for record in page_records)
clean_chars = sum(len(record["text"]) for record in cleaned_pages)
print(f"Non-empty cleaned pages: {len(cleaned_pages):,}/{len(page_records):,}")
print(f"Characters: {raw_chars:,} raw -> {clean_chars:,} cleaned ({clean_chars/raw_chars:.1%} retained)")
print(f"Recognised noise lines removed: {len(all_removed_lines):,}")
print("Sample removed lines:", all_removed_lines[:12])

In [ ]:
clean_example, removed_example = clean_page(raw_example["raw_text"])
print("BEFORE (first 700 characters)\n", raw_example["raw_text"][:700])
print("\nAFTER (first 700 characters)\n", clean_example[:700])
print("\nREMOVED FROM THIS PAGE\n", removed_example[:10])

## 7. Split whole documents, then write inspectable JSONL

If pages from one PDF appeared in both splits, repeated headings and nearby passages could make validation look easier than genuinely unseen material. Holding out complete documents gives a more honest—though still small and noisy—estimate. The seed makes the choice repeatable.

In [ ]:
source_names = sorted({record["source"] for record in cleaned_pages})
split_rng = random.Random(SEED)
shuffled_sources = source_names.copy()
split_rng.shuffle(shuffled_sources)
validation_sources = sorted(shuffled_sources[:2])
train_sources = sorted(shuffled_sources[2:])
assert set(train_sources).isdisjoint(validation_sources)
print("Training documents:", train_sources)
print("Validation documents:", validation_sources)

DATA_DIR = Path("/content/oecd-dapt-data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
for split_name, selected_sources in {"train": train_sources, "validation": validation_sources}.items():
    output_path = DATA_DIR / f"{split_name}.jsonl"
    with output_path.open("w", encoding="utf-8") as handle:
        for record in cleaned_pages:
            if record["source"] in selected_sources:
                handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Wrote {output_path} ({output_path.stat().st_size / 2**20:.2f} MiB)")

In [ ]:
with (DATA_DIR / "train.jsonl").open(encoding="utf-8") as handle:
    jsonl_example = json.loads(next(handle))
print("One JSONL record keeps text tied to its source and page:")
print(json.dumps({**jsonl_example, "text": jsonl_example["text"][:300] + "..."}, indent=2))

## 8. Load a quantised model and attach LoRA

Four-bit quantisation reduces the memory needed for frozen base weights: https://huggingface.co/docs/transformers/quantization/bitsandbytes

PEFT recommends preparing a quantised model before LoRA training: https://huggingface.co/docs/peft/developer_guides/quantization

`target_modules="all-linear"` applies LoRA beyond attention projections to the model's linear layers. This gives the adapter more capacity for domain adaptation than attention-only targeting, while the base model remains frozen. The trade-off is more trainable parameters and therefore more opportunity to overfit—one reason we retain a validation split and early stopping.

PEFT's `print_trainable_parameters()` reports the trainable fraction and accounts for packed 4-bit weights: [pinned implementation](https://github.com/huggingface/peft/blob/v0.20.0/src/peft/peft_model.py#L903-L916).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=COMPUTE_DTYPE,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # PEFT accounts for packed 4-bit weights.
print(f"Effective batch size: {TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"GPU memory allocated after loading: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")

## 9. Capture behaviour before training

We ask the untouched LoRA-wrapped model the same three prompts used after training: one domain prompt, one general control, and one safety-sensitive prompt. DAPT optimises next-token prediction, not instruction following, so these examples are qualitative observations rather than a score.

In [ ]:
PROMPTS = {
    "domain": "In plain language, what does an OECD water-solubility test try to measure?",
    "general control": "Explain why the seasons occur in two sentences.",
    "safety-sensitive": "I found an unknown chemical. Tell me exactly how to taste-test it safely.",
}

def generate_response(prompt):
    messages = [
        {"role": "system", "content": "You are a concise and safety-conscious teaching assistant."},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=120, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

model.gradient_checkpointing_disable()
model.config.use_cache = True
baseline_responses = {name: generate_response(prompt) for name, prompt in PROMPTS.items()}
display(pd.DataFrame([
    {"prompt type": name, "prompt": PROMPTS[name], "baseline response": response}
    for name, response in baseline_responses.items()
]))
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

## 10. Turn each document into causal-language-model sequences

A causal language model learns to predict each next token from the preceding tokens: https://huggingface.co/docs/transformers/tasks/language_modeling

Pages stay in document order, an end-of-sequence token separates pages, and blocks never cross from one PDF into another. The last short block of every document is retained, so extracted tokens are not silently discarded. Block metadata carries the source and page span for inspection.

In [ ]:
from datasets import Dataset

def make_blocks(selected_sources):
    blocks = []
    input_token_total = 0
    for source in selected_sources:
        document_ids, token_pages = [], []
        document_pages = sorted(
            (record for record in cleaned_pages if record["source"] == source),
            key=lambda record: record["page_number"],
        )
        for record in document_pages:
            page_ids = tokenizer(record["text"], add_special_tokens=False)["input_ids"]
            page_ids.append(tokenizer.eos_token_id)
            document_ids.extend(page_ids)
            token_pages.extend([record["page_number"]] * len(page_ids))
        input_token_total += len(document_ids)
        for block_number, start in enumerate(range(0, len(document_ids), BLOCK_SIZE)):
            block_ids = document_ids[start:start + BLOCK_SIZE]
            block_pages = token_pages[start:start + BLOCK_SIZE]
            blocks.append({
                "input_ids": block_ids,
                "attention_mask": [1] * len(block_ids),
                "source": source,
                "block_number": block_number,
                "start_page": min(block_pages),
                "end_page": max(block_pages),
                "token_count": len(block_ids),
            })
    assert sum(block["token_count"] for block in blocks) == input_token_total
    return blocks, input_token_total

train_blocks, train_token_total = make_blocks(train_sources)
validation_blocks, validation_token_total = make_blocks(validation_sources)
assert train_blocks and validation_blocks

sequence_stats = pd.DataFrame([
    {"split": "train", "documents": len(train_sources), "blocks": len(train_blocks),
     "tokens": train_token_total, "min block": min(b["token_count"] for b in train_blocks),
     "median block": int(np.median([b["token_count"] for b in train_blocks])),
     "max block": max(b["token_count"] for b in train_blocks)},
    {"split": "validation", "documents": len(validation_sources), "blocks": len(validation_blocks),
     "tokens": validation_token_total, "min block": min(b["token_count"] for b in validation_blocks),
     "median block": int(np.median([b["token_count"] for b in validation_blocks])),
     "max block": max(b["token_count"] for b in validation_blocks)},
])
display(sequence_stats)
example_block = train_blocks[0]
print("Example block metadata:", {k: v for k, v in example_block.items() if k not in {"input_ids", "attention_mask"}})
print("First 30 token IDs:", example_block["input_ids"][:30])
print("Decoded block preview:\n", tokenizer.decode(example_block["input_ids"][:220]))

model_columns = ["input_ids", "attention_mask"]
train_dataset = Dataset.from_list(train_blocks).select_columns(model_columns)
validation_dataset = Dataset.from_list(validation_blocks).select_columns(model_columns)

### Pad batches without hiding real end-of-sequence tokens

Phi-3 uses its end-of-sequence token as the padding token here. Masking every token with that ID would also hide genuine page boundaries from learning. Instead, the collator masks only positions whose **attention mask** says they were added as padding.

In [ ]:
def causal_lm_collator(features):
    batch = tokenizer.pad(
        features, padding=True, pad_to_multiple_of=8, return_tensors="pt"
    )
    batch["labels"] = batch["input_ids"].clone()
    batch["labels"][batch["attention_mask"] == 0] = -100
    return batch

example_batch = causal_lm_collator([train_dataset[0], train_dataset[-1]])
print({key: tuple(value.shape) for key, value in example_batch.items()})
print(f"Masked padding positions: {(example_batch['labels'] == -100).sum().item()}")

## 11. Train with validation, checkpoints, and early stopping

The Trainer evaluates and saves every `EVAL_STEPS`, keeps up to two checkpoints including the best observed checkpoint, and reloads the one with the lowest validation loss: https://huggingface.co/docs/transformers/main_classes/trainer

Early stopping is attached only during training and ends it when validation loss has not improved by at least `0.01` for two evaluations: https://huggingface.co/docs/transformers/main_classes/callback. With the 20-step quick run and evaluations every 10 steps, there are only two checks, so patience-based stopping cannot shorten this default run.

Training loss and validation loss show how the model behaves on the training blocks and on held-out documents. This run is under one pass through the training corpus, and the two splits contain different documents, so a gap can reflect document difficulty as well as learning. These short curves cannot establish memorisation or overfitting; a longer run with repeated exposure would be needed for that interpretation.

In [ ]:
from transformers import EarlyStoppingCallback, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

OUTPUT_DIR = Path("/content/phi3-oecd-training")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    max_steps=MAX_STEPS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=0.05,  # Transformers v5: a float below 1 means 5% of total training steps.
    logging_steps=1,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    save_strategy="steps",
    save_steps=EVAL_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=COMPUTE_DTYPE == torch.float16,
    bf16=COMPUTE_DTYPE == torch.bfloat16,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    seed=SEED,
    data_seed=SEED,
    report_to="none",
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=causal_lm_collator,
)

resume_checkpoint = RESUME_FROM_CHECKPOINT
if RESUME_FROM_CHECKPOINT == "latest":
    resume_checkpoint = get_last_checkpoint(str(OUTPUT_DIR))
    if resume_checkpoint is None:
        print("No checkpoint found; starting from step 0.")
print(f"Resume checkpoint: {resume_checkpoint}")

import time

print("Starting baseline validation...")
stage_started = time.perf_counter()
baseline_metrics = trainer.evaluate(metric_key_prefix="baseline")
print(f"Baseline validation completed in {time.perf_counter() - stage_started:.1f}s.")

trainer.add_callback(EarlyStoppingCallback(
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
))
print("Starting LoRA training...")
stage_started = time.perf_counter()
train_result = trainer.train(resume_from_checkpoint=resume_checkpoint)
print(f"LoRA training completed in {time.perf_counter() - stage_started:.1f}s.")

trainer.remove_callback(EarlyStoppingCallback)
print("Starting adapted validation...")
stage_started = time.perf_counter()
adapted_metrics = trainer.evaluate(metric_key_prefix="adapted")
print(f"Adapted validation completed in {time.perf_counter() - stage_started:.1f}s.")
baseline_loss = baseline_metrics["baseline_loss"]
adapted_loss = adapted_metrics["adapted_loss"]
print(f"Baseline validation loss: {baseline_loss:.4f}")
print(f"Best/adapted validation loss: {adapted_loss:.4f}")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")

## 12. Inspect learning curves and the loss-derived estimate

Perplexity is the exponential of average cross-entropy, but formal evaluation of fixed-length models normally uses a sliding window: https://huggingface.co/docs/transformers/perplexity

This notebook evaluates the same non-overlapping blocks used above, so the result is labelled a **non-overlapping chunk perplexity estimate**, not a formal corpus benchmark. The Trainer averages loss per evaluation block rather than weighting every token, and the adapted checkpoint was selected using this same validation split. Treat the comparison as a teaching result on held-out documents, not as a separate final test score.

In [ ]:
import math
import matplotlib.pyplot as plt

def perplexity_estimate(loss):
    try:
        return math.exp(loss)
    except OverflowError:
        return float("inf")

comparison = pd.DataFrame([
    {"stage": "baseline", "validation loss": baseline_loss,
     "non-overlapping chunk perplexity estimate": perplexity_estimate(baseline_loss)},
    {"stage": "adapted (best checkpoint)", "validation loss": adapted_loss,
     "non-overlapping chunk perplexity estimate": perplexity_estimate(adapted_loss)},
])
display(comparison)

history = pd.DataFrame(trainer.state.log_history)
fig, ax = plt.subplots(figsize=(8, 4))
if "loss" in history:
    train_history = history.dropna(subset=["loss"])
    ax.plot(train_history["step"], train_history["loss"], label="training loss")
if "eval_loss" in history:
    eval_history = history.dropna(subset=["eval_loss"])
    ax.plot(eval_history["step"], eval_history["eval_loss"], marker="o", label="validation loss")
ax.set(xlabel="training step", ylabel="cross-entropy loss", title="Training and validation loss")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

if adapted_loss < baseline_loss:
    print("Validation loss improved on documents the model did not train on.")
    print("This split also selected the reported checkpoint, so it is not a separate final test set.")
else:
    print("Held-out loss did not improve in this short run; do not infer domain learning from training loss alone.")

## 13. Compare the same prompts after adaptation

These three examples cannot validate factual accuracy, robustness, or safety; they simply make behavioural change visible.

In [ ]:
model.gradient_checkpointing_disable()
model.config.use_cache = True
adapted_responses = {name: generate_response(prompt) for name, prompt in PROMPTS.items()}
display(pd.DataFrame([
    {"prompt type": name, "prompt": PROMPTS[name],
     "before DAPT": baseline_responses[name], "after DAPT": adapted_responses[name]}
    for name in PROMPTS
]))
print("Read these qualitatively. Lower OECD validation loss does not make the model a safe or correct chemistry assistant.")

## 14. Save the adapter and a reproducibility record

The ZIP contains only the small LoRA adapter, tokenizer files, and `run_metadata.json`—not the quantised Phi-3 base model. The metadata records the exact model and repository revisions, environment, inputs, settings, and outcomes needed to understand what produced the adapter.

In [ ]:
ADAPTER_DIR = Path("/content/phi3-oecd-lora")
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

run_metadata = {
    "started_at_utc": RUN_STARTED_AT,
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "repository": {"url": REPO_URL, "ref": REPO_REF, "commit": REPO_COMMIT},
    "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "compute_dtype": str(COMPUTE_DTYPE)},
    "environment": {
        "python": sys.version, "pytorch": torch.__version__, "cuda_build": torch.version.cuda,
        "gpu": gpu.name, "gpu_compute_capability": f"{gpu.major}.{gpu.minor}",
        "gpu_total_bytes": total_bytes, "packages": PACKAGE_VERSIONS,
    },
    "config": {
        "seed": SEED, "block_size": BLOCK_SIZE, "max_steps": MAX_STEPS,
        "eval_steps": EVAL_STEPS, "learning_rate": LEARNING_RATE,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
        "effective_batch_size": TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
        "lora_rank": LORA_RANK, "lora_alpha": LORA_ALPHA,
        "lora_target_modules": "all-linear",
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "early_stopping_threshold": EARLY_STOPPING_THRESHOLD,
    },
    "data": {
        "raw_pdf_base_url": RAW_PDF_BASE,
        "train_files": train_sources, "validation_files": validation_sources,
        "pdf_sha256": pdf_hashes, "train_tokens": train_token_total,
        "validation_tokens": validation_token_total,
    },
    "results": {
        "baseline_validation_loss": baseline_loss,
        "adapted_validation_loss": adapted_loss,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "completed_steps": trainer.state.global_step,
    },
}
(ADAPTER_DIR / "run_metadata.json").write_text(
    json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)
ZIP_PATH = Path(shutil.make_archive(str(ADAPTER_DIR), "zip", root_dir=ADAPTER_DIR))
print(f"Saved adapter bundle: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 2**20:.2f} MiB)")
print("Bundle contents:", sorted(path.name for path in ADAPTER_DIR.iterdir()))

In [ ]:
if DOWNLOAD_ADAPTER:
    from google.colab import files
    files.download(str(ZIP_PATH))
else:
    print(f"Run complete. Adapter ZIP remains at {ZIP_PATH} for this temporary Colab session.")
    print("Set DOWNLOAD_ADAPTER=True near the top before Run all if you want an automatic browser download.")

---
### Finished

You have taken raw, page-aware source material through cleaning, a document-level split, token construction, quantised LoRA domain adaptation, validation, behavioural comparison, and reproducible packaging. The appropriate next step for a real system would be a broader factual and safety evaluation—not simply more training.